In [ ]:
import numpy as np
import plotly.graph_objects as go

# 全局图对象，用于连续添加椭球
_current_fig = None


def ellipsoid(
    a=1,
    b=1,
    c=1,
    center=(0, 0, 0),
    rotation=None,
    affine_matrix=None,
    u_res=30,
    v_res=30,
    u_range=(0, 2 * np.pi),
    v_range=(0, np.pi),
    color="rgba(100,150,255,0.7)",
    wireframe=False,
    name="Ellipsoid",
    show_axes=False,
    fig=None,
    auto_add=True,
    **kwargs,
):
    """
    绘制椭球。

    参数：
        a, b, c: 椭球三个方向的半轴长度（默认 1 为球体）
        center: 椭球中心 (x, y, z)
        rotation: 旋转角度 (rx, ry, rz)（欧拉角，度数），None 表示无旋转
        affine_matrix: 可选 4x4 齐次仿射矩阵，优先于 rotation/center
        u_res, v_res: 网格分辨率（越大越精细，默认 30）
        u_range: u 参数范围 (u_min, u_max)，默认 (0, 2π) 为完整圆周
        v_range: v 参数范围 (v_min, v_max)，默认 (0, π) 为完整球面
        color: 椭球颜色（支持 rgba、hex、named color）
        wireframe: True 时绘制网格线而非实心
        name: 图例名称
        show_axes: True 时显示坐标轴
        fig: 指定 Figure 对象，None 时使用全局图或创建新的
        auto_add: 如果 True 且 fig=None，自动使用全局图（连续添加）；
                 如果 False，创建新图（默认 True）
        **kwargs: 其他 go.Surface 参数（如 hovertemplate）

    返回：
        fig (go.Figure)

    说明：
        - 新参数 `affine_matrix`：可选，4x4 齐次仿射矩阵（numpy array）。
          如果提供，则优先使用该矩阵对点做仿射变换（包括旋转、缩放、平移、镜像等），
          此时 `rotation` 与 `center` 将被忽略（它们可以先被用来构造 affine 矩阵）。
        - 保持原有的 `u_range` / `v_range` 用于绘制部分椭球面。
    """
    global _current_fig

    # 参数化椭球面
    u = np.linspace(u_range[0], u_range[1], u_res)
    v = np.linspace(v_range[0], v_range[1], v_res)
    u_grid, v_grid = np.meshgrid(u, v)

    # 基础椭球坐标
    x = a * np.cos(u_grid) * np.sin(v_grid)
    y = b * np.sin(u_grid) * np.sin(v_grid)
    z = c * np.cos(v_grid)

    # 旋转/仿射处理
    pts = np.stack([x, y, z], axis=-1)
    shape = pts.shape
    pts_flat = pts.reshape(-1, 3)

    if affine_matrix is not None:
        # 期望 affine_matrix 为 4x4 齐次矩阵
        A = np.asarray(affine_matrix)
        if A.shape == (3, 3):
            # 仅线性部分 -> 转为 4x4（无平移）
            A4 = np.eye(4)
            A4[:3, :3] = A
            A = A4
        if A.shape == (3, 4):
            A4 = np.eye(4)
            A4[:3, :4] = A
            A = A4
        if A.shape != (4, 4):
            raise ValueError("affine_matrix must be shape (4,4), (3,3) or (3,4)")

        ones = np.ones((pts_flat.shape[0], 1))
        pts_h = np.concatenate([pts_flat, ones], axis=1)  # (N,4)
        pts_trans = pts_h @ A.T
        pts_trans_reshaped = pts_trans[:, :3].reshape(shape)
        x, y, z = (
            pts_trans_reshaped[..., 0],
            pts_trans_reshaped[..., 1],
            pts_trans_reshaped[..., 2],
        )
    else:
        # 仍然支持按欧拉角旋转并平移到 center（原行为）
        if rotation:
            rx, ry, rz = (
                np.radians(rotation[0]),
                np.radians(rotation[1]),
                np.radians(rotation[2]),
            )
            Rx = np.array(
                [[1, 0, 0], [0, np.cos(rx), -np.sin(rx)], [0, np.sin(rx), np.cos(rx)]]
            )
            Ry = np.array(
                [[np.cos(ry), 0, np.sin(ry)], [0, 1, 0], [-np.sin(ry), 0, np.cos(ry)]]
            )
            Rz = np.array(
                [[np.cos(rz), -np.sin(rz), 0], [np.sin(rz), np.cos(rz), 0], [0, 0, 1]]
            )
            R = Rz @ Ry @ Rx

            pts_rot = pts_flat @ R.T
            pts_rotated = pts_rot.reshape(shape)
            x, y, z = pts_rotated[..., 0], pts_rotated[..., 1], pts_rotated[..., 2]

        # 平移到中心
        x = x + center[0]
        y = y + center[1]
        z = z + center[2]

    # 确定要使用的 Figure
    if fig is None:
        if auto_add and _current_fig is not None:
            fig = _current_fig  # 使用全局图
        else:
            fig = go.Figure()  # 创建新图

    # 添加 Surface
    if wireframe:
        # 网格模式：只显示线条
        for i in range(v_res):
            fig.add_trace(
                go.Scatter3d(
                    x=x[i],
                    y=y[i],
                    z=z[i],
                    mode="lines",
                    line=dict(color=color, width=1),
                    showlegend=(i == 0),
                    name=name,
                    hoverinfo="skip",
                )
            )
        for j in range(u_res):
            fig.add_trace(
                go.Scatter3d(
                    x=x[:, j],
                    y=y[:, j],
                    z=z[:, j],
                    mode="lines",
                    line=dict(color=color, width=1),
                    showlegend=False,
                    hoverinfo="skip",
                )
            )
    else:
        # 实心模式：使用 color 参数创建均匀颜色
        surface_kwargs = {
            k: v for k, v in kwargs.items() if k not in ["u_range", "v_range"]
        }

        # 创建均匀的 surfacecolor（所有点使用相同颜色）
        surfacecolor = np.ones_like(z)

        fig.add_trace(
            go.Surface(
                x=x,
                y=y,
                z=z,
                surfacecolor=surfacecolor,
                colorscale=[[0, color], [1, color]],
                showscale=False,
                name=name,
                **surface_kwargs,
            )
        )

    # 设置坐标轴
    axis_dict = dict(showgrid=True, zeroline=show_axes)
    fig.update_layout(
        scene=dict(
            xaxis=axis_dict, yaxis=axis_dict, zaxis=axis_dict, aspectmode="data"
        ),
        hovermode="closest",
    )

    return fig


def start_figure(title="Ellipsoids", width=900, height=700):
    """
    启动一个新的全局图，用于连续添加椭球。

    参数：
        title: 图表标题
        width: 图表宽度
        height: 图表高度

    返回：
        fig (go.Figure)

    使用示例：
        fig = start_figure("My Ellipsoids")
        ellipsoid(a=1, name='Red')
        ellipsoid(a=2, center=(3,0,0), name='Blue')
        show_figure(fig)
    """
    global _current_fig
    _current_fig = go.Figure()
    _current_fig.update_layout(
        title=title,
        width=width,
        height=height,
        showlegend=True,
        scene=dict(aspectmode="data"),
    )
    return _current_fig


def show_figure(fig=None):
    """
    显示当前图或指定的图。

    参数：
        fig: 要显示的 Figure，None 时显示全局图
    """
    global _current_fig
    if fig is None:
        fig = _current_fig
    if fig is not None:
        fig.show()
    else:
        print("No figure to show. Use start_figure() first.")


def clear_figure():
    """清空全局图。"""
    global _current_fig
    _current_fig = None


def ellipsoids(*ellipsoid_args, title="Ellipsoids", figsize=(900, 700)):
    """
    一次性绘制多个椭球。

    参数：
        *ellipsoid_args: 可变数量的椭球参数字典，每个字典传给 ellipsoid()
        title: 图标题
        figsize: (width, height)

    示例：
        fig = ellipsoids(
            {'a': 1, 'b': 2, 'c': 0.5, 'color': 'red'},
            {'a': 1.5, 'b': 1, 'c': 2, 'center': (2, 0, 0), 'color': 'blue'}
        )
    """
    fig = go.Figure()
    for args in ellipsoid_args:
        ellipsoid(**args, fig=fig)

    fig.update_layout(title=title, width=figsize[0], height=figsize[1], showlegend=True)
    return fig


# 演示基础用法
# fig1 = ellipsoid(
#     a=4, b=1, c=1, u_range=(0, np.pi), v_range=(0, np.pi / 2), auto_add=True
# )
fig1 = ellipsoid(
    a=4,
    b=1,
    c=1,
    u_range=(0, np.pi),
    v_range=(0, np.pi / 2),
    affine_matrix=np.diag([-1.0, 1.0, -1.0, 1.0]),
    auto_add=True,
    color="rgba(255,0,0,0.7)",
)
# fig1 = ellipsoids({"a": 1, "b": 2, "c": 0.5, "color": "red"})
fig1.show()

In [ ]:
# 演示：部分椭球、旋转、仿射变换（平移、镜像/缩放）
import numpy as np

# 1) 部分椭球（上半球）
fig_a = ellipsoids({
    'a': 1.0,
    'b': 1.0,
    'c': 1.0,
    'color': 'rgba(255,100,100,0.8)',
})
# 直接用 v_range 绘制半球
fig_a = go.Figure()
ellipsoid(a=1, b=1, c=1, v_range=(0, np.pi/2), color='rgba(255,100,100,0.8)', name='半球', fig=fig_a)
fig_a.update_layout(title='部分椭球（上半球）', width=700, height=500)
fig_a.show()

# 2) 使用 rotation 参数旋转椭球
fig_b = go.Figure()
ellipsoid(a=1, b=2, c=0.7, rotation=(30, 45, 0), color='rgba(100,150,255,0.8)', name='旋转椭球', fig=fig_b)
fig_b.update_layout(title='使用 rotation（欧拉角）', width=700, height=500)
fig_b.show()

# 3) 使用 affine_matrix 做平移（等价于先绘制再平移）
T = np.eye(4)
T[:3, 3] = np.array([3.0, 0.0, 0.0])  # 沿 x 轴平移 3
fig_c = go.Figure()
ellipsoid(a=1, b=1.5, c=0.6, affine_matrix=T, color='rgba(120,220,120,0.8)', name='平移椭球', fig=fig_c)
fig_c.update_layout(title='仿射平移（affine_matrix）', width=700, height=500)
fig_c.show()

# 4) 使用 affine_matrix 做镜像（在 x 轴上做负缩放）
A = np.diag([-1.0, 1.0, 1.0, 1.0])  # x 轴镜像
# 可叠加平移：先镜像再平移到可见区域
A[:3, 3] = np.array([3.0, 0.0, 0.0])
fig_d = go.Figure()
ellipsoid(a=1, b=0.6, c=1.2, affine_matrix=A, color='rgba(200,150,255,0.8)', name='镜像椭球', fig=fig_d)
fig_d.update_layout(title='镜像（x 轴翻转）+ 平移', width=700, height=500)
fig_d.show()

# 5) 组合示例：同一图中叠加多个仿射变换后的椭球
fig_combo = start_figure('组合仿射示例', width=1000, height=700)
# 原始
ellipsoid(a=1, b=1, c=1, color='rgba(180,180,180,0.4)', name='原始', fig=fig_combo)
# 平移
ellipsoid(a=0.7, b=1.2, c=0.6, affine_matrix=T, color='rgba(120,220,120,0.6)', name='平移', fig=fig_combo)
# 镜像
ellipsoid(a=0.8, b=0.8, c=1.1, affine_matrix=A, color='rgba(200,150,255,0.6)', name='镜像', fig=fig_combo)
fig_combo.show()


In [ ]:
import numpy as np
import plotly.graph_objects as go

# 全局图对象，用于连续添加椭球
_current_fig = None


def ellipsoid_fixed(
    a=1,
    b=1,
    c=1,
    center=(0, 0, 0),
    rotation=None,
    affine_matrix=None,
    u_res=30,
    v_res=30,
    u_range=(0, 2 * np.pi),
    v_range=(0, np.pi),
    color="rgba(100,150,255,0.7)",
    wireframe=False,
    name="Ellipsoid",
    show_axes=False,
    fig=None,
    auto_add=True,
    **kwargs,
):
    """
    绘制椭球（修复颜色版本）。

    参数：
        a, b, c: 椭球三个方向的半轴长度（默认 1 为球体）
        center: 椭球中心 (x, y, z)
        rotation: 旋转角度 (rx, ry, rz)（欧拉角，度数），None 表示无旋转
        affine_matrix: 可选 4x4 齐次仿射矩阵，优先于 rotation/center
        u_res, v_res: 网格分辨率（越大越精细，默认 30）
        u_range: u 参数范围 (u_min, u_max)，默认 (0, 2π) 为完整圆周
        v_range: v 参数范围 (v_min, v_max)，默认 (0, π) 为完整球面
        color: 椭球颜色（支持 rgba、hex、named color）
        wireframe: True 时绘制网格线而非实心
        name: 图例名称
        show_axes: True 时显示坐标轴
        fig: 指定 Figure 对象，None 时使用全局图或创建新的
        auto_add: 如果 True 且 fig=None，自动使用全局图（连续添加）；
                 如果 False，创建新图（默认 True）
        **kwargs: 其他 go.Surface 参数（如 hovertemplate）

    返回：
        fig (go.Figure)
    """
    global _current_fig

    # 参数化椭球面
    u = np.linspace(u_range[0], u_range[1], u_res)
    v = np.linspace(v_range[0], v_range[1], v_res)
    u_grid, v_grid = np.meshgrid(u, v)

    # 基础椭球坐标
    x = a * np.cos(u_grid) * np.sin(v_grid)
    y = b * np.sin(u_grid) * np.sin(v_grid)
    z = c * np.cos(v_grid)

    # 旋转/仿射处理
    pts = np.stack([x, y, z], axis=-1)
    shape = pts.shape
    pts_flat = pts.reshape(-1, 3)

    if affine_matrix is not None:
        # 期望 affine_matrix 为 4x4 齐次矩阵
        A = np.asarray(affine_matrix)
        if A.shape == (3, 3):
            # 仅线性部分 -> 转为 4x4（无平移）
            A4 = np.eye(4)
            A4[:3, :3] = A
            A = A4
        if A.shape == (3, 4):
            A4 = np.eye(4)
            A4[:3, :4] = A
            A = A4
        if A.shape != (4, 4):
            raise ValueError("affine_matrix must be shape (4,4), (3,3) or (3,4)")

        ones = np.ones((pts_flat.shape[0], 1))
        pts_h = np.concatenate([pts_flat, ones], axis=1)  # (N,4)
        pts_trans = pts_h @ A.T
        pts_trans_reshaped = pts_trans[:, :3].reshape(shape)
        x, y, z = (
            pts_trans_reshaped[..., 0],
            pts_trans_reshaped[..., 1],
            pts_trans_reshaped[..., 2],
        )
    else:
        # 仍然支持按欧拉角旋转并平移到 center（原行为）
        if rotation:
            rx, ry, rz = (
                np.radians(rotation[0]),
                np.radians(rotation[1]),
                np.radians(rotation[2]),
            )
            Rx = np.array(
                [[1, 0, 0], [0, np.cos(rx), -np.sin(rx)], [0, np.sin(rx), np.cos(rx)]]
            )
            Ry = np.array(
                [[np.cos(ry), 0, np.sin(ry)], [0, 1, 0], [-np.sin(ry), 0, np.cos(ry)]]
            )
            Rz = np.array(
                [[np.cos(rz), -np.sin(rz), 0], [np.sin(rz), np.cos(rz), 0], [0, 0, 1]]
            )
            R = Rz @ Ry @ Rx

            pts_rot = pts_flat @ R.T
            pts_rotated = pts_rot.reshape(shape)
            x, y, z = pts_rotated[..., 0], pts_rotated[..., 1], pts_rotated[..., 2]

        # 平移到中心
        x = x + center[0]
        y = y + center[1]
        z = z + center[2]

    # 确定要使用的 Figure
    if fig is None:
        if auto_add and _current_fig is not None:
            fig = _current_fig  # 使用全局图
        else:
            fig = go.Figure()  # 创建新图

    # 添加 Surface
    if wireframe:
        # 网格模式：只显示线条
        for i in range(v_res):
            fig.add_trace(
                go.Scatter3d(
                    x=x[i],
                    y=y[i],
                    z=z[i],
                    mode="lines",
                    line=dict(color=color, width=1),
                    showlegend=(i == 0),
                    name=name,
                    hoverinfo="skip",
                )
            )
        for j in range(u_res):
            fig.add_trace(
                go.Scatter3d(
                    x=x[:, j],
                    y=y[:, j],
                    z=z[:, j],
                    mode="lines",
                    line=dict(color=color, width=1),
                    showlegend=False,
                    hoverinfo="skip",
                )
            )
    else:
        # 实心模式：修复颜色显示问题
        surface_kwargs = {
            k: v for k, v in kwargs.items() if k not in ["u_range", "v_range"]
        }

        # 方法1：直接使用 colorscale（推荐）
        fig.add_trace(
            go.Surface(
                x=x,
                y=y,
                z=z,
                colorscale=[[0, color], [1, color]],  # 使用单一颜色
                showscale=False,
                name=name,
                cmin=0,
                cmax=1,
                **surface_kwargs,
            )
        )

    # 设置坐标轴
    axis_dict = dict(showgrid=True, zeroline=show_axes)
    fig.update_layout(
        scene=dict(
            xaxis=axis_dict, yaxis=axis_dict, zaxis=axis_dict, aspectmode="data"
        ),
        hovermode="closest",
    )

    return fig


def start_figure(title="Ellipsoids", width=900, height=700):
    """
    启动一个新的全局图，用于连续添加椭球。

    参数：
        title: 图表标题
        width: 图表宽度
        height: 图表高度

    返回：
        fig (go.Figure)
    """
    global _current_fig
    _current_fig = go.Figure()
    _current_fig.update_layout(
        title=title,
        width=width,
        height=height,
        showlegend=True,
        scene=dict(aspectmode="data"),
    )
    return _current_fig


def show_figure(fig=None):
    """
    显示当前图或指定的图。

    参数：
        fig: 要显示的 Figure，None 时显示全局图
    """
    global _current_fig
    if fig is None:
        fig = _current_fig
    if fig is not None:
        fig.show()
    else:
        print("No figure to show. Use start_figure() first.")


def clear_figure():
    """清空全局图。"""
    global _current_fig
    _current_fig = None


# 测试颜色修复
def test_colors():
    """测试不同颜色的椭球"""
    fig = start_figure("颜色测试")
    matrix1 = np.eye(4)
    matrix1[2, 3] = -2
    matrix2 = np.eye(4)
    matrix2[0, 0] = -1.0
    matrix2[2, 2] = -1.0
    matrix2[2, 3] = -0.6
    # 测试红色
    ellipsoid_fixed(
        a=3,
        b=1,
        c=1,
        center=(0, 0, 0),
        u_res=10,
        v_res=10,
        u_range=(0, np.pi),
        v_range=(0, np.pi / 2 / 2),
        color="rgba(255,0,0,0.7)",
        affine_matrix=matrix1,
        wireframe=True,
        name="红色",
    )

    # 测试绿色
    ellipsoid_fixed(
        a=3,
        b=1,
        c=1,
        center=(0, 0, 0),
        u_res=10,
        v_res=10,
        u_range=(0, np.pi),
        v_range=(0, np.pi / 2 / 2),
        affine_matrix=matrix2,
        color="rgba(0,255,0,0.7)",
        wireframe=True,
        name="绿色",
    )

    # 测试蓝色
    # ellipsoid_fixed(
    #     a=2, b=2, c=2, center=(10, 0, 0), color="rgba(0,0,255,0.7)", name="蓝色"
    # )

    # # 测试黄色
    # ellipsoid_fixed(
    #     a=2, b=2, c=2, center=(15, 0, 0), color="rgba(255,255,0,0.7)", name="黄色"
    # )

    show_figure(fig)


# 单独测试一个红色椭球（使用你原来的参数）
def test_red_ellipsoid():
    """测试红色椭球（使用你原来的参数）"""
    fig = ellipsoid_fixed(
        a=4,
        b=1,
        c=1,
        u_range=(0, np.pi),
        v_range=(0, np.pi / 2),
        affine_matrix=np.diag([-1.0, 1.0, -1.0, 1.0]),
        auto_add=True,
        color="rgba(255,0,0,0.7)",
    )
    fig.show()


if __name__ == "__main__":
    # 运行测试
    # print("测试颜色修复...")
    test_colors()
    # print("红色椭球测试...")
    # test_red_ellipsoid()

In [ ]:
# 修改版本：在参数化椭球面时限制Y轴正方向（变换后y >= 0），然后生成网格
import numpy as np
import plotly.graph_objects as go

# 参数设置
a = 4  # 椭球x半轴
b = 1  # 椭球y半轴
c = 1  # 椭球z半轴
u_res = 30  # u方向分辨率
v_res = 30  # v方向分辨率
translation_y = -0.5  # 沿Y轴负方向平移量

# 计算变换后y >= 0所需的y最小值（变换前）
y_min_required = -translation_y  # 因为 y_trans = y + translation_y >= 0 => y >= -translation_y

# y = b * sin(u) * sin(v)，要y >= y_min_required
# 由于u_range=(0, pi)，sin(u) >= 0；v_range需要调整使 sin(v) >= y_min_required / b
v_min = np.arcsin(max(0, y_min_required / b))  # 确保v_min >= 0
v_max = np.pi / 2  # 上半球

u_range = (0, np.pi)
v_range = (v_min, v_max)

print(f"调整后的v_range: ({v_min:.3f}, {v_max:.3f})，确保变换前y >= {y_min_required:.3f}")

# 构造仿射变换矩阵：沿Y轴平移
affine_matrix = np.eye(4)  # 4x4单位矩阵
affine_matrix[1, 3] = translation_y  # Y轴平移

# 参数化椭球面（已限制在Y轴正方向）
u = np.linspace(u_range[0], u_range[1], u_res)
v = np.linspace(v_range[0], v_range[1], v_res)
u_grid, v_grid = np.meshgrid(u, v)

# 计算基础椭球坐标
x = a * np.cos(u_grid) * np.sin(v_grid)
y = b * np.sin(u_grid) * np.sin(v_grid)
z = c * np.cos(v_grid)

# 应用仿射变换
pts = np.stack([x, y, z], axis=-1)  # 堆叠为(N, N, 3)
shape = pts.shape
pts_flat = pts.reshape(-1, 3)  # 展平为(N*N, 3)
ones = np.ones((pts_flat.shape[0], 1))  # 添加齐次坐标
pts_h = np.concatenate([pts_flat, ones], axis=1)  # (N*N, 4)
pts_trans = pts_h @ affine_matrix.T  # 应用变换
pts_trans_reshaped = pts_trans[:, :3].reshape(shape)  # 恢复形状
x_trans, y_trans, z_trans = (
    pts_trans_reshaped[..., 0],
    pts_trans_reshaped[..., 1],
    pts_trans_reshaped[..., 2],
)

# 调试：打印Y坐标的范围
print(f"变换后Y坐标范围: min={y_trans.min():.3f}, max={y_trans.max():.3f}")
print(f"所有点Y >= 0: {np.all(y_trans >= 0)}")

# 创建Plotly图表
fig = go.Figure()

# 使用Surface显示曲面（所有点都可见，因为已限制Y>=0）
fig.add_trace(
    go.Surface(
        x=x_trans,
        y=y_trans,
        z=z_trans,
        colorscale=[[0, "rgba(255,0,0,0.7)"], [1, "rgba(255,0,0,0.7)"]],  # 红色
        showscale=False,  # 不显示颜色条
        name="限制Y正方向椭球曲面",
        cmin=0,
        cmax=1,
    )
)

# 设置图表布局
fig.update_layout(
    title=f"参数化时限制Y轴正方向的椭球（平移{translation_y}）",
    scene=dict(
        xaxis=dict(showgrid=True, title="X轴"),
        yaxis=dict(showgrid=True, title="Y轴"),
        zaxis=dict(showgrid=True, title="Z轴"),
        aspectmode="data",  # 保持数据比例
    ),
    hovermode="closest",  # 悬停模式
    width=900,  # 图表宽度
    height=700,  # 图表高度
)

# 显示图表
fig.show()